<a href="https://colab.research.google.com/github/lucas-estanislau/pe-de-meia/blob/Teste-de-otimiza%C3%A7%C3%A3o-da-coleta/pe_de_meia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Pé-de-Meia**
### e sua relação com a permanência e o desempenho escolar no ensino médio no Brasil

In [23]:
import pandas as pd
import numpy as np
import requests
import zipfile
from io import BytesIO

In [24]:
# Carrega os dados zipados de cada mês do programa dentro do Portal da Transparência
def carregar_mes(url):
  response = requests.get(url, timeout=120)
  response.raise_for_status()

  with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
      arquivo = zip_ref.namelist()[0]

      with zip_ref.open(arquivo) as csv:
          df = pd.read_csv(csv, sep=";", encoding="latin1", usecols=["UF", "VALOR PARCELA"])

  return df

# Converte 'VALOR PARCELA' de object para float
def converter_valor(df):
  df['VALOR PARCELA'] = (
      df['VALOR PARCELA'].str.replace(",", ".", regex=False).astype(float)
  )
  return df

# Colera os dados do referido mês, trata a coluna de 'VALOR PARCELA' e agrupar por UF. Devolve um df resumido e agrupado
def coletar_mes(url):
  df_mes = carregar_mes(url)
  converter_valor(df_mes)
  df_resumido = df_mes.groupby('UF')['VALOR PARCELA'].sum().reset_index().sort_values('VALOR PARCELA', ascending=False)

  return df_resumido

In [25]:
# Definindo as URL para coletar os arquivos do Portal da Transparência
mar24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202403"
abr24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202404"
mai24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202405"
jun24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202406"
ago24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202408"
set24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202409"
out24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202410"
nov24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202411"
dez24 = "https://portaldatransparencia.gov.br/download-de-dados/pe-de-meia/202412"

# Lista com as variáveis que guardam as URLs
dfs = [mar24,
       abr24,
       mai24,
       jun24,
       ago24,
       set24,
       out24,
       nov24,
       dez24]

# Lista para armazenar os DFs agrupados por UF com soma do valor recebido por cada UF
dfs_resumidos = []

# Coleta os dados de todos os meses resumidos do ano de 2024 e armazena em 'dfs_rsumidos'
for mes in dfs:
  df_resumido = coletar_mes(mes)
  dfs_resumidos.append(df_resumido)

In [26]:
# Lista com os nomes dos meses para salvar
meses = ["mar24", "abr24", "mai24", "jun24", "ago24", "set24", "out24", "nov24", "dez24"
]

# Salva cada DF de acordo com o seu mês
for mes, df in zip(meses, dfs_resumidos):
    df.to_csv(
        f"/content/drive/MyDrive/pe-de-meia/pe-de-meia-data/2024/{mes}_uf.csv",
        index=False
    )